# Cross-Industry Accelerator — Create Dashboards
### Auto-Create Real-Time KQL Dashboard + Power BI Report

This notebook creates **two dashboard types** for the selected industry:

| Type | Technology | Data Source | Use Case |
|---|---|---|---|
| **Real-Time Dashboard** | Fabric KQL Dashboard | Eventhouse | Live operational monitoring |
| **Static Report** | Power BI Report | Semantic Model / Warehouse | Historical analysis & KPIs |

**What it does:**
1. Reads industry configuration and auto-discovered schemas
2. Generates KQL queries for real-time tiles (event streams, anomalies, trends)
3. Creates a Fabric Real-Time Dashboard with auto-generated pages and tiles
4. Creates a Power BI Report shell linked to the semantic model

> **Prerequisites:**
> 1. Run `02_Load_Lakehouse.ipynb` for Eventhouse data (real-time dashboard)
> 2. Run `04_Create_Semantic_Model.ipynb` for the Power BI report
> 3. Eventhouse must be configured for real-time tiles

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════
%run ./00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# AUTO-GENERATE KQL QUERIES FOR REAL-TIME TILES (ZT-hardened)
# ════════════════════════════════════════════════════════════════════════

import json, requests, base64
import sempy.fabric as fabric
import pandas as pd

# Build KQL queries from event/streaming table schemas
kql_tiles = []

for table_name in EVENTHOUSE_TABLES:
    csv_path = f"{CSV_BASE_PATH}/{table_name}.csv"
    try:
        df = spark.read.option("header", True).option("inferSchema", True).csv(csv_path)
        kql_table = table_name.replace("fact_", "").replace("stream_", "")

        # ZT: Validate KQL table name
        try:
            sanitize_identifier(kql_table)
        except ValueError as e:
            print(f"  ⚠ ZT: Skipping unsafe KQL table name: {kql_table!r}")
            log_audit_event("KQL_TILE_REJECTED", kql_table, str(e), "BLOCKED")
            continue

        # ZT: Validate all column names before interpolating into KQL queries
        all_cols = [f.name for f in df.schema.fields]
        safe_col_set = set(sanitize_columns(all_cols))

        cols = [c for c in all_cols if c in safe_col_set]

        # Detect timestamp and numeric columns for dashboarding (using validated names only)
        ts_cols = [c for c in cols if any(kw in c.lower() for kw in ["timestamp", "datetime", "date", "time", "created", "_at"])]
        num_cols = [f.name for f in df.schema.fields
                    if str(f.dataType) in ("IntegerType()", "LongType()", "FloatType()", "DoubleType()")
                    and not f.name.lower().endswith("_id")
                    and f.name in safe_col_set]
        cat_cols = [f.name for f in df.schema.fields
                    if str(f.dataType) == "StringType()" and not f.name.lower().endswith("_id")
                    and not any(kw in f.name.lower() for kw in ["name", "description", "notes", "comment"])
                    and f.name in safe_col_set]

        ts_col = ts_cols[0] if ts_cols else None

        # Tile 1: Event count time-series (last 24h)
        if ts_col:
            kql_tiles.append({
                "title": f"{kql_table} — Events Over Time",
                "query": f"{kql_table}\n| where {ts_col} > ago(24h)\n| summarize EventCount=count() by bin({ts_col}, 15m)\n| order by {ts_col} asc",
                "visualization": "timechart",
                "table": kql_table,
            })

        # Tile 2: Breakdown by category (top 10)
        if cat_cols:
            cat_col = cat_cols[0]
            kql_tiles.append({
                "title": f"{kql_table} — By {cat_col}",
                "query": f"{kql_table}\n| summarize Count=count() by {cat_col}\n| top 10 by Count desc",
                "visualization": "piechart",
                "table": kql_table,
            })

        # Tile 3: Numeric KPI (avg of first numeric column)
        if num_cols and ts_col:
            num_col = num_cols[0]
            label = num_col.replace("_", " ").title()
            kql_tiles.append({
                "title": f"{kql_table} — Avg {label} Trend",
                "query": f"{kql_table}\n| where {ts_col} > ago(24h)\n| summarize Avg_{num_col}=avg({num_col}) by bin({ts_col}, 1h)\n| order by {ts_col} asc",
                "visualization": "linechart",
                "table": kql_table,
            })

        # Tile 4: Latest events (live feed)
        kql_tiles.append({
            "title": f"{kql_table} — Latest Events",
            "query": f"{kql_table}\n| top 20 by {ts_col if ts_col else cols[0]} desc",
            "visualization": "table",
            "table": kql_table,
        })

        log_audit_event("KQL_TILES_GENERATED", kql_table,
            f"{sum(1 for t in kql_tiles if t['table'] == kql_table)} tiles")

    except Exception as e:
        print(f"  ⚠ Could not generate tiles for {table_name}: {e}")
        log_audit_event("KQL_TILE_ERROR", table_name, str(e), "ERROR")

print(f"✅ Generated {len(kql_tiles)} KQL dashboard tiles from {len(EVENTHOUSE_TABLES)} event tables.")
print(f"   (ZT: All column/table names validated before KQL interpolation)")
for tile in kql_tiles:
    print(f"   [{tile['visualization']:<10}] {tile['title']}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CREATE FABRIC REAL-TIME KQL DASHBOARD
# ════════════════════════════════════════════════════════════════════════

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')

RT_DASHBOARD_NAME = f"{INDUSTRY_LABEL}_RealTime_Dashboard"

if not EVENTHOUSE_CLUSTER_URI or not EVENTHOUSE_DATABASE:
    print("⚠ Eventhouse not configured — skipping real-time dashboard creation.")
    print("  Configure EVENTHOUSE_CLUSTER_URI and EVENTHOUSE_DATABASE in 00_Industry_Config.")
else:
    # Build dashboard definition with pages and tiles
    # Group tiles by source table into pages
    pages_by_table = {}
    for tile in kql_tiles:
        tbl = tile["table"]
        if tbl not in pages_by_table:
            pages_by_table[tbl] = []
        pages_by_table[tbl].append(tile)

    dashboard_pages = []
    for page_name, tiles in pages_by_table.items():
        page_tiles = []
        for i, tile in enumerate(tiles):
            page_tiles.append({
                "id": f"tile_{page_name}_{i}",
                "title": tile["title"],
                "query": tile["query"],
                "visualType": tile["visualization"],
                "dataSourceUri": EVENTHOUSE_CLUSTER_URI,
                "database": EVENTHOUSE_DATABASE,
                "layout": {"x": (i % 2) * 6, "y": (i // 2) * 4, "width": 6, "height": 4},
            })
        dashboard_pages.append({
            "id": f"page_{page_name}",
            "name": page_name.replace("_", " ").title(),
            "tiles": page_tiles,
        })

    dashboard_definition = {
        "dataSourceUri": EVENTHOUSE_CLUSTER_URI,
        "database": EVENTHOUSE_DATABASE,
        "pages": dashboard_pages,
        "autoRefresh": {"enabled": True, "intervalMs": 30000},
    }

    # Create via Fabric REST API
    FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"
    url = f"{FABRIC_API_BASE}/workspaces/{workspace_id}/items"
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }

    dash_b64 = base64.b64encode(json.dumps(dashboard_definition).encode("utf-8")).decode("utf-8")

    payload = {
        "displayName": RT_DASHBOARD_NAME,
        "description": f"Real-time operational dashboard for {INDUSTRY}",
        "type": "KQLDashboard",
        "definition": {
            "parts": [{
                "path": "dashboard.json",
                "payload": dash_b64,
                "payloadType": "InlineBase64"
            }]
        }
    }

    print(f"Creating real-time dashboard: {RT_DASHBOARD_NAME}...")
    resp = requests.post(url, headers=headers, json=payload)

    if resp.status_code in (200, 201, 202):
        print(f"\n✅ Real-Time Dashboard created: {RT_DASHBOARD_NAME}")
        resp_json = resp.json()
        if "id" in resp_json:
            print(f"   Item ID: {resp_json['id']}")
        print(f"   Pages:   {len(dashboard_pages)}")
        print(f"   Tiles:   {sum(len(p['tiles']) for p in dashboard_pages)}")
    else:
        print(f"\n⚠ Dashboard creation returned status {resp.status_code}")
        print(f"   Response: {resp.text}")
        print(f"\n   The dashboard definition JSON has been generated above.")
        print(f"   You can manually create the dashboard and paste the KQL queries.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CREATE POWER BI REPORT (STATIC DASHBOARD)
# ════════════════════════════════════════════════════════════════════════

REPORT_NAME = f"{INDUSTRY_LABEL}_Analytics_Report"

# Resolve semantic model ID
items_df = fabric.list_items()
sm_matches = items_df[(items_df["Type"] == "SemanticModel") & (items_df["Display Name"] == SEMANTIC_MODEL_NAME)]

if sm_matches.empty:
    print(f"⚠ Semantic model '{SEMANTIC_MODEL_NAME}' not found.")
    print(f"   Run 04_Create_Semantic_Model.ipynb first.")
    print(f"   Available semantic models:")
    print(items_df[items_df["Type"] == "SemanticModel"][["Display Name", "Id"]].to_string(index=False))
else:
    sm_id = str(sm_matches.iloc[0].Id)
    print(f"✓ Semantic model: {SEMANTIC_MODEL_NAME} → {sm_id}")

    # Build report definition: pages for dim overview + fact KPIs
    report_pages = []

    # Page 1: Executive Summary
    report_pages.append({
        "name": "Executive Summary",
        "displayName": "Executive Summary",
        "visuals": [
            {"type": "card", "title": "Key Metrics", "description": "High-level KPIs from fact tables"},
            {"type": "clusteredBarChart", "title": "Category Breakdown", "description": "Top categories across dimensions"},
        ]
    })

    # Page 2+: One page per major fact table
    fact_tables_all = FACT_TABLES_BATCH + FACT_TABLES_EVENT
    for ft in fact_tables_all[:5]:  # limit to 5 pages
        label = ft.replace("fact_", "").replace("_", " ").title()
        report_pages.append({
            "name": ft,
            "displayName": label,
            "visuals": [
                {"type": "table", "title": f"{label} Detail", "source": ft},
                {"type": "lineChart", "title": f"{label} Trend", "source": ft},
                {"type": "clusteredBarChart", "title": f"{label} by Category", "source": ft},
            ]
        })

    report_definition = {
        "datasetId": sm_id,
        "pages": report_pages,
    }

    report_b64 = base64.b64encode(json.dumps(report_definition).encode("utf-8")).decode("utf-8")

    payload = {
        "displayName": REPORT_NAME,
        "description": f"Analytics report for {INDUSTRY} industry",
        "type": "Report",
        "definition": {
            "parts": [{
                "path": "report.json",
                "payload": report_b64,
                "payloadType": "InlineBase64"
            }]
        }
    }

    print(f"\nCreating Power BI report: {REPORT_NAME}...")
    resp = requests.post(url, headers=headers, json=payload)

    if resp.status_code in (200, 201, 202):
        print(f"\n✅ Report created: {REPORT_NAME}")
        resp_json = resp.json()
        if "id" in resp_json:
            print(f"   Item ID: {resp_json['id']}")
        print(f"   Pages:   {len(report_pages)}")
    else:
        print(f"\n⚠ Report creation returned status {resp.status_code}")
        print(f"   Response: {resp.text}")
        print(f"\n   You can create the report manually in Power BI using the semantic model.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# KQL QUERY REFERENCE — COPY/PASTE FOR MANUAL DASHBOARD CREATION
# ════════════════════════════════════════════════════════════════════════
# If the API-based dashboard creation didn't work, use these KQL queries
# to manually create tiles in a Fabric Real-Time Dashboard.

print(f"{'═'*70}")
print(f"KQL QUERY REFERENCE — {INDUSTRY.upper()}")
print(f"{'═'*70}")
print(f"Database: {EVENTHOUSE_DATABASE}")
print(f"Cluster:  {EVENTHOUSE_CLUSTER_URI}\n")

for i, tile in enumerate(kql_tiles, 1):
    print(f"── Tile {i}: {tile['title']} ({tile['visualization']}) ──")
    print(tile['query'])
    print()

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# DASHBOARD CREATION SUMMARY
# ════════════════════════════════════════════════════════════════════════

print(f"\n{'═'*65}")
print(f"DASHBOARD SUMMARY — {INDUSTRY.upper()}")
print(f"{'═'*65}")
print(f"")
print(f"  Real-Time Dashboard:  {RT_DASHBOARD_NAME if EVENTHOUSE_CLUSTER_URI else 'N/A (Eventhouse not configured)'}")
if EVENTHOUSE_CLUSTER_URI:
    print(f"    Data Source:        {EVENTHOUSE_CLUSTER_URI}")
    print(f"    Database:           {EVENTHOUSE_DATABASE}")
    print(f"    Pages:              {len(pages_by_table) if 'pages_by_table' in dir() else 0}")
    print(f"    KQL Tiles:          {len(kql_tiles)}")
    print(f"    Auto-Refresh:       30 seconds")
print(f"")
print(f"  Power BI Report:      {REPORT_NAME if 'REPORT_NAME' in dir() else 'N/A'}")
if 'report_pages' in dir():
    print(f"    Semantic Model:     {SEMANTIC_MODEL_NAME}")
    print(f"    Pages:              {len(report_pages)}")
print(f"")
print(f"{'═'*65}")
print(f"✅ All dashboards created. The {INDUSTRY} accelerator pipeline is complete!")
print(f"")
print(f"  Pipeline Summary:")
print(f"  00 → Industry Config")
print(f"  01 → Data Ingestion & Discovery")
print(f"  02 → Lakehouse + Eventhouse Loading")
print(f"  03 → Warehouse Loading")
print(f"  04 → Semantic Model Creation")
print(f"  05 → Ontology Creation")
print(f"  06 → Data Agent Creation")
print(f"  07 → Dashboard Creation ← YOU ARE HERE")